## Convert DP2 light curves to embeddings

In this notebook we will calculate the embeddings for ECDFS and EDFS - two DDFs (Deep Drilling Field).

In [ ]:
import lsdb
import numpy as np
import pandas as pd

from dask.distributed import Client

In [2]:
dia_object = lsdb.open_catalog('/astro/store/shire/hats/dash/hats/v30_0_6/dia_object_collection')

Let's specify the Deep Drilling Fields of Rubin DP1:

In [3]:
fields = {
    "ECDFS": (53.13, -28.10),  # Extended Chandra Deep Field South
    "EDFS": (59.10, -48.73),  # Euclid Deep Field South
    "Rubin_SV_38_7": (37.86, 6.98),  # Low Ecliptic Latitude Field
    "Rubin_SV_95_-25": (95.00, -25.00),  # Low Galactic Latitude Field
    "47_Tuc": (6.02, -72.08),  # 47 Tuc Globular Cluster
    "Fornax_dSph": (40.00, -34.45),  # Fornax Dwarf Spheroidal Galaxy
}

# Red for extragalactic fields, orange for dense fields
field_styles = {
    "ECDFS": ("red", "solid"),
    "EDFS": ("red", "dashed"),
    "Rubin_SV_38_7": ("red", "dotted"),
    "Rubin_SV_95_-25": ("orange", "solid"),
    "47_Tuc": ("orange", "dashed"),
    "Fornax_dSph": ("orange", "dotted"),
}

# Define a 4-degree search radius
radius_arcsec = 4 * 3600

# Create six cone searches
cones = {name: lsdb.ConeSearch(ra=ra, dec=dec, radius_arcsec=radius_arcsec) for name, (ra, dec) in fields.items()}

Let's identify which DDFs have coverage in DP2:

In [4]:
fig, ax = dia_object.plot_pixels()
for name, cone in cones.items():
    color, linestyle = field_styles[name]
    cone.plot(ax=ax, ec=f"tab:{color}", linestyle=linestyle, label=name)
    ax.legend()

Only "ECDFS" and "EDFS" seem to be covered. 

In [5]:
# I tried to concat both fields in a single catalog but that failed
dia_object_path = '/astro/store/shire/hats/dash/hats/v30_0_6/dia_object_collection'
columns = ["diaObjectId","ra","dec","diaSource"]
ecdfs = lsdb.open_catalog(dia_object_path, columns=columns, search_filter=cones["ECDFS"])
edfs = lsdb.open_catalog(dia_object_path, columns=columns, search_filter=cones["EDFS"])
ecdfs.concat(edfs)

In [6]:
# Get an approximation with box search
box = lsdb.BoxSearch(ra=[50,70], dec=[-60,-20])
cat = lsdb.open_catalog(dia_object_path, columns=columns, search_filter=box)
cat.plot_pixels()

We will filter out any bad detections:

In [7]:
flag_cols = [c for c in cat.meta["diaSource"].columns if 'flag' in c.lower()]
query_str = "not (" + " or ".join(f"diaSource.{c}" for c in flag_cols) + ")"
cat = cat.query(query_str).query("diaSource.len() > 100")

### Generate field embeddings

Using the `light_curve` package. ATCAT processes all six LSST ugrizY bands jointly and returns 384-dimensional embeddings. Inputs are flux (AB, zero-point 31.4 by default), flux error, time, and integer band index (u=0, g=1, r=2, i=3, z=4, Y=5).

In [8]:
import onnxruntime as ort
from light_curve.embed import ATCAT

def get_model(provider):
    """Instantiates the ATCAT model for inference"""
    return ATCAT.from_hf(
        output="last",
        band_groups={"u": 0, "g": 1, "r": 2, "i": 3, "z": 4, "y": 5},
        ort_session_kwargs={"providers": [provider], "sess_options": _session_options()},
    )

def _session_options():
    """ONNX Runtime will try to pin threads to specific CPU cores. 
    This helps fix the affinity warnings."""
    sess_options = ort.SessionOptions()
    sess_options.intra_op_num_threads = 1
    sess_options.inter_op_num_threads = 1
    return sess_options

def compute_embeddings(time, flux, flux_err, band, *, provider):
    """Calculate embeddings for a lightcurve"""
    model = get_model(provider)
    emb = model(time, flux, flux_err, band)
    return {"embeddings.value": emb.flatten()}

### CPU vs GPU

#### CPU

We will start with CPU. Pass the CPU provider accordingly:

In [9]:
embeddings = cat.map_rows(
    compute_embeddings,
    columns=[
        "diaSource.midpointMjdTai",
        "diaSource.psfFlux",
        "diaSource.psfFluxErr",
        "diaSource.band"
    ],
    provider="CPUExecutionProvider",
    row_container="args",
    append_columns=True,
    meta={"embeddings.value": np.float32},
)

When we have an empty partition `map_rows` fails with metadata mismatch. Hack below:

In [10]:
def _hack(df, *, provider):
    if len(df) == 0:
        df["embeddings.value"] = pd.Series([], dtype=np.float32)
        return df
    return df.map_rows(
        compute_embeddings, 
        columns=["diaSource.midpointMjdTai","diaSource.psfFlux","diaSource.psfFluxErr","diaSource.band"], 
        provider=provider,
        row_container="args",
        append_columns=True
    )

embeddings = cat.map_partitions(lambda df: _hack(df, provider="CPUExecutionProvider"))
embeddings

In [11]:
with Client(n_workers=4):
    embeddings.write_catalog("outputs/ddf_embeddings", overwrite=True)

#### GPU

We already installed the `onnxruntime-gpu` variant with pixi. Pass the provider accordingly. For example, for NVIDIA GPUs:

```python
model = ATCAT.from_hf(output="last", ort_session_kwargs={"providers": ["CUDAExecutionProvider"]})
```

In [12]:
embeddings = cat.map_partitions(lambda df: _hack(df, provider="CUDAExecutionProvider"))
embeddings

For the GPU part of the workflow I needed to install the right CUDA libraries. Thanks Kostya for the huge help!

In [ ]:
with Client(n_workers=4):
    embeddings.write_catalog("outputs/ddf_embeddings", overwrite=True)

### Progress utilities

```bash
# RSP (CPU only):
## Dask Dashboard
https://usdf-rsp.slac.stanford.edu/nb/user/<username>/proxy/8787/status

# Gondor:
## Dask Dashboard -> access localhost:<local_port> after port forwarding
ssh -L -N <local_port>:localhost:<remote_port> <username>@gondor
## GPU usage
watch -n 1 nvidia-smi
```

In the next notebook we will load all the embeddings and get similarity matches for each of the First Look targets.